In [56]:
#初始合并数据用代码
import os, zipfile
import pandas as pd
import numpy as np
from collections import defaultdict


ZIP_PATH = "D:/project/Features_group_10.zip"           
EXTRACT_DIR = "D:/project/features_extracted"
FOLDER_NAME = "Features_group_10"
OUTCOMES_PATH = "D:/project/Outcomes-group_10.txt"
OUTPUT_CSV = "D:/project/icu_features.csv"

STATIC_KEYS = {"RecordID", "Age", "Gender", "Height", "Weight", "ICUType"}

def time_to_minutes(tstr: str) -> int:
    try:
        hh, mm = tstr.split(":")
        return int(hh) * 60 + int(mm)
    except Exception:
        return 0

def parse_patient_file(filepath: str):
    static = {}
    dynamic = defaultdict(list)
    with open(filepath, "r") as f:
        header = f.readline() 
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 3:
                continue
            t, param, val = parts[0], parts[1], parts[2]
            if t == "00:00" and param in STATIC_KEYS:
                try:
                    static[param] = float(val) if param != "RecordID" else int(val)
                except:
                    static[param] = np.nan
            else:
                tm = time_to_minutes(t)
                try:
                    v = float(val)
                except:
                    v = np.nan
                dynamic[param].append((tm, v))
    dyn_df_dict = {}
    for p, lst in dynamic.items():
        if not lst:
            continue
        dfp = pd.DataFrame(lst, columns=["time_min", "value"]).dropna(subset=["value"])
        if not dfp.empty:
            dyn_df_dict[p] = dfp.sort_values("time_min").reset_index(drop=True)
    return static, dyn_df_dict

def aggregate_dynamic(dyn_df_dict: dict):
    feats = {}
    for param, dfp in dyn_df_dict.items():
        if dfp.empty:
            continue
        vals = dfp["value"]
        feats[f"{param}__mean"]  = float(vals.mean())
        feats[f"{param}__min"]   = float(vals.min())
        feats[f"{param}__max"]   = float(vals.max())
        feats[f"{param}__std"]   = float(vals.std(ddof=0)) if len(vals) > 1 else 0.0
        feats[f"{param}__last"]  = float(dfp.iloc[-1]["value"])
        feats[f"{param}__count"] = int(len(vals))
    return feats

def main():
    if not os.path.exists(os.path.join(EXTRACT_DIR, FOLDER_NAME)):
        if os.path.exists(ZIP_PATH):
            with zipfile.ZipFile(ZIP_PATH, "r") as zf:
                zf.extractall(EXTRACT_DIR)
        else:
            os.makedirs(EXTRACT_DIR, exist_ok=True)
    DATA_DIR = os.path.join(EXTRACT_DIR, FOLDER_NAME)

    rows = []
    files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".txt")])
    for i, fname in enumerate(files, 1):
        fpath = os.path.join(DATA_DIR, fname)
        static, dyn = parse_patient_file(fpath)
        if "RecordID" not in static:
            try:
                static["RecordID"] = int(os.path.splitext(fname)[0])
            except:
                continue
        feats = aggregate_dynamic(dyn)
        rows.append({**static, **feats})
        if i % 200 == 0:
            print(f"Processed {i}/{len(files)} files...")

    features_df = pd.DataFrame(rows)
    if "RecordID" in features_df.columns:
        features_df["RecordID"] = features_df["RecordID"].astype(int, errors="ignore")

    # Outcomes
    outcomes = pd.read_csv(OUTCOMES_PATH)
    outcomes.columns = [c.strip() for c in outcomes.columns]
    for col in outcomes.columns:
        if col == "In-hospital_death":
            continue
        outcomes[col] = outcomes[col].replace(-1, np.nan)

    merged = features_df.merge(outcomes, on="RecordID", how="inner")

    merged = merged.drop(columns=["Length_of_stay", "Survival"], errors="ignore")

    merged.to_csv(OUTPUT_CSV, index=False)
    print("Saved:", OUTPUT_CSV, "with shape", merged.shape)

if __name__ == "__main__":
    main()


In [ ]:
#初步数据探索分析
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Config
CSV_PATH = r"D:/project/icu_features.csv"  
OUT_DIR  = "D:/project/icu_eda_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print("Columns:", len(df.columns))


# Basic summary 
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
desc = df[num_cols].describe().T
print("\n=== 数值特征统计描述 ===")
print(desc.head(20))  

# 缺失值情况
na_cnt = df.isna().sum().sort_values(ascending=False)
na_rate = (df.isna().mean().sort_values(ascending=False) * 100).round(2)
missing_df = pd.DataFrame({"na_count": na_cnt, "na_rate_%": na_rate})
print("\n=== 缺失值情况（前20列） ===")
print(missing_df.head(20))

def save_show(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)


# 1) 标签分布图
if "In-hospital_death" in df.columns:
    fig = plt.figure(figsize=(5,4))
    labels, counts = np.unique(df["In-hospital_death"].dropna().values, return_counts=True)
    plt.bar([str(int(x)) for x in labels], counts)
    plt.title("Class Distribution (0=Survive, 1=Death)")
    plt.xlabel("In-hospital_death")
    plt.ylabel("Count")
    save_show(fig, "class_distribution.png")

    ratios = df["In-hospital_death"].value_counts(normalize=True).sort_index()
    print("\n=== 标签分布 ===")
    print(df["In-hospital_death"].value_counts())
    print("比例：")
    print(ratios)


# 2) 年龄分布（按存活/死亡）
if "Age" in df.columns and "In-hospital_death" in df.columns:
    fig = plt.figure(figsize=(7,4))
    alive = df[df["In-hospital_death"] == 0]["Age"].dropna().values
    dead  = df[df["In-hospital_death"] == 1]["Age"].dropna().values
    bins = 20
    plt.hist(alive, bins=bins, alpha=0.6, label="Survive (0)")
    plt.hist(dead,  bins=bins, alpha=0.6, label="Death (1)")
    plt.title("Age Distribution by Outcome")
    plt.xlabel("Age")
    plt.ylabel("Count")
    plt.legend()
    save_show(fig, "age_distribution_by_outcome.png")


# 3) ICUType 分布（按标签分组）
if "ICUType" in df.columns and "In-hospital_death" in df.columns:
    ctab = pd.crosstab(df["ICUType"], df["In-hospital_death"]).sort_index()
    icu_vals = ctab.index.tolist()
    x = np.arange(len(icu_vals))
    fig = plt.figure(figsize=(7,4))
    width = 0.35

    if 0 in ctab.columns:
        plt.bar(x - width/2, ctab[0].values, width, label="Survive (0)")
    if 1 in ctab.columns:
        plt.bar(x + width/2, ctab[1].values, width, label="Death (1)")

    plt.xticks(x, [str(v) for v in icu_vals])
    plt.title("ICU Type Distribution by Outcome")
    plt.xlabel("ICUType")
    plt.ylabel("Count")
    plt.legend()
    save_show(fig, "icu_type_distribution_by_outcome.png")


# 4) SOFA vs SAPS-I 散点
if ("SAPS-I" in df.columns) and ("SOFA" in df.columns) and ("In-hospital_death" in df.columns):
    fig = plt.figure(figsize=(6,5))
    mask0 = df["In-hospital_death"] == 0
    mask1 = df["In-hospital_death"] == 1
    plt.scatter(df.loc[mask0, "SAPS-I"], df.loc[mask0, "SOFA"], s=12, alpha=0.6, label="Survive (0)")
    plt.scatter(df.loc[mask1, "SAPS-I"], df.loc[mask1, "SOFA"], s=12, alpha=0.6, label="Death (1)")
    plt.xlabel("SAPS-I")
    plt.ylabel("SOFA")
    plt.title("SOFA vs SAPS-I by Outcome")
    plt.legend()
    save_show(fig, "severity_scatter_saps_vs_sofa.png")



In [8]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============ 基础路径设置 ==============
DATA_PATH = r"D:/project/icu_features.csv"
OUTDIR = r"D:/project/eda"
TARGET = "In-hospital_death"
os.makedirs(OUTDIR, exist_ok=True)

# ============ 读取并预处理 ==============
df = pd.read_csv(DATA_PATH)
leaky_cols = ["RecordID", "Survival", "ICU_length_of_stay", "Readmission"]
df = df.drop(columns=[c for c in leaky_cols if c in df.columns])
df = df.apply(pd.to_numeric, errors='ignore')

# ============ 1) 缺失率分析 ==============
missing = df.isna().mean().sort_values(ascending=False) * 100
missing.to_csv(os.path.join(OUTDIR, "missing_rate.csv"))
plt.figure(figsize=(10,8))
missing.head(30).plot(kind='bar')
plt.title("Top 30 Missing Rate (%)")
plt.ylabel("Missing %")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "missing_rate_top30.png"))
plt.close()

# ============ 2) 死亡 vs 存活统计 =========
df[TARGET] = df[TARGET].astype(int)
desc = df.groupby(TARGET).describe().T
desc.to_csv(os.path.join(OUTDIR, "desc_by_outcome.csv"))


# ============ 3) 连续变量选择 =============
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != TARGET]
numeric_cols = [c for c in numeric_cols if df[c].nunique() > 1]
num_plot_cols = numeric_cols
# ============ 4) 箱线图 =============
n_cols = 4
n_rows = int(np.ceil(len(num_plot_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*4))
axes = axes.flatten()
for ax, col in zip(axes, num_plot_cols):
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Value")
for j in range(len(num_plot_cols), len(axes)):
    fig.delaxes(axes[j])
plt.suptitle("Boxplots by Outcome (Raw Data)", fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "boxplots_combined.png"), dpi=150, bbox_inches="tight")
plt.close()


# 5) KDE 密度图（用直方图近似密度）

num_plot_cols = numeric_cols   

n = len(num_plot_cols)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for idx, col in enumerate(num_plot_cols):
    ax = axes[idx]

    series = df[col].dropna()
    if series.nunique() <= 1:
        ax.set_title(f"{col} (constant)")
        ax.set_xticks([])
        continue
    for cls, style in zip([0, 1], ["solid", "dashed"]):
        subset = df[df[TARGET] == cls][col].dropna()
        if len(subset) > 5:
            ax.hist(
                subset,
                bins=30,
                density=True,
                histtype="step",
                linestyle=style,
                linewidth=1.5,
                label=f"{TARGET}={cls}"
            )

    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

# 删除多余空子图
for j in range(len(num_plot_cols), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Density (Histogram Approx.) by Outcome", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(os.path.join(OUTDIR, "kde_combined.png"), dpi=150)
plt.close()

# ============ 6) Spearman 相关矩阵 ==========
top_corr_cols = numeric_cols[:30]
corr_mat = df[top_corr_cols].corr(method='spearman')
corr_mat.to_csv(os.path.join(OUTDIR, "spearman_corr.csv"))
plt.figure(figsize=(10,8))
plt.imshow(corr_mat, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(top_corr_cols)), top_corr_cols, rotation=90)
plt.yticks(range(len(top_corr_cols)), top_corr_cols)
plt.title("Spearman Correlation Heatmap (Raw Data)")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "spearman_heatmap.png"))
plt.close()



✅ 缺失率分析完成 -> missing_rate.csv / missing_rate_top30.png
✅ 分组描述统计完成 -> desc_by_outcome.csv
✅ 箱线图完成 -> boxplots_combined.png
✅ KDE 图（直方图近似）已生成 -> kde_combined.png
✅ 相关矩阵完成 -> spearman_corr.csv / spearman_heatmap.png

🎉 全部EDA完成，结果保存在: D:/project/eda


In [2]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np
import os

DATA_PATH = r"D:/project/icu_features.csv"
OUTDIR = r"D:/project/eda_raw"
os.makedirs(OUTDIR, exist_ok=True)

TARGET = "In-hospital_death"

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["RecordID"])

# 选数值列
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# 去掉目标 & 完全缺失列
num_cols = [c for c in num_cols if c != TARGET and df[c].notna().sum() > 0]

df_vif = df[num_cols].fillna(df[num_cols].median())

# 计算 VIF
vif_data = []
for i, col in enumerate(df_vif.columns):
    vif = variance_inflation_factor(df_vif.values, i)
    vif_data.append([col, vif])

vif_df = pd.DataFrame(vif_data, columns=["feature", "VIF"]).sort_values("VIF", ascending=False)
vif_df.to_csv(os.path.join(OUTDIR, "vif_results.csv"), index=False)

display(vif_df.head(10))


D:\a\lib\site-packages\statsmodels\regression\linear_model.py:1715: RuntimeWarning: divide by zero encountered in double_scalars
  return 1 - self.ssr/self.centered_tss
D:\a\lib\site-packages\statsmodels\regression\linear_model.py:1715: RuntimeWarning: invalid value encountered in double_scalars
  return 1 - self.ssr/self.centered_tss
D:\a\lib\site-packages\statsmodels\stats\outliers_influence.py:193: RuntimeWarning: divide by zero encountered in double_scalars
  vif = 1. / (1. - r_squared_i)


,feature,VIF
216,Cholesterol__min,inf
218,Cholesterol__std,inf
215,Cholesterol__mean,inf
217,Cholesterol__max,inf
34,DiasABP__count,2.436241e+04
58,SysABP__count,2.435988e+04
133,WBC__max,1.995912e+04
134,WBC__std,1.929767e+04
219,Cholesterol__last,2.391822e+03
7,pH__max,2.264048e+03


In [53]:
import pandas as pd
p = r"D:/project/icu_outputs"
df = pd.read_csv(p + "/train_selected.csv")
df2 = pd.read_csv(p + "/test_selected.csv")
full = pd.concat([df, df2], ignore_index=True)
full.to_csv(p + "/full_selected.csv", index=False)
print("saved:", p + "/full_selected.csv", full.shape)

saved: D:/project/icu_outputs/full_selected.csv (4000, 40)


In [ ]:
import os
import json
import argparse
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET = "In-hospital_death"
ID_COL = "RecordID"

def stratified_split(df: pd.DataFrame, y_col: str, test_size: float, random_state: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    y = df[y_col]
    X = df.drop(columns=[y_col])
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(sss.split(X, y))
    return df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

def compute_missing_signal(train_df: pd.DataFrame, feature_cols: List[str], target_col: str) -> pd.DataFrame:
    y = train_df[target_col]
    rows = []
    for col in feature_cols:
        miss_flag = train_df[col].isna()
        m_rate = float(miss_flag.mean())
        if m_rate == 0.0 or m_rate == 1.0:
            continue
        dm = float(y[miss_flag].mean()) if miss_flag.sum() > 0 else np.nan
        dn = float(y[~miss_flag].mean()) if (~miss_flag).sum() > 0 else np.nan
        if np.isfinite(dm) and np.isfinite(dn):
            rows.append([col, m_rate, dm, dn, dm - dn])
    out = pd.DataFrame(rows, columns=["feature","missing_rate_train","death_rate_missing","death_rate_not_missing","difference"])\
            .sort_values("difference", ascending=False)
    return out

def add_missing_flags(train_df: pd.DataFrame, test_df: pd.DataFrame, feature_cols: List[str], threshold: float):
    miss_rate = train_df[feature_cols].isna().mean()
    high_missing = miss_rate[miss_rate > threshold].index.tolist()
    for col in high_missing:
        train_df[f"{col}_missing"] = train_df[col].isna().astype(int)
        test_df[f"{col}_missing"]  = test_df[col].isna().astype(int)
    return train_df, test_df, high_missing

def impute_train_test(train_df: pd.DataFrame, test_df: pd.DataFrame, feature_cols: List[str]):
    num_cols = train_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in feature_cols if c not in num_cols]
    # median fit on TRAIN
    med = train_df[num_cols].median()
    train_df[num_cols] = train_df[num_cols].fillna(med)
    test_df[num_cols]  = test_df[num_cols].fillna(med)
    # most-frequent for categoricals
    for c in cat_cols:
        mode_val = train_df[c].mode(dropna=True)
        fill_val = mode_val.iloc[0] if len(mode_val) else ""
        train_df[c] = train_df[c].fillna(fill_val)
        test_df[c]  = test_df[c].fillna(fill_val)
    return train_df, test_df, med.to_dict()

def plot_roc_pr(y_true, proba, prefix, outdir):
    fpr, tpr, _ = roc_curve(y_true, proba)
    auc = roc_auc_score(y_true, proba)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{prefix} ROC")
    plt.legend(loc="lower right")
    rp = os.path.join(outdir, f"{prefix}_roc.png")
    plt.savefig(rp, dpi=150, bbox_inches="tight"); plt.close()

    prec, rec, _ = precision_recall_curve(y_true, proba)
    ap = average_precision_score(y_true, proba)
    plt.figure()
    plt.plot(rec, prec, label=f"AP={ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"{prefix} PR")
    plt.legend(loc="lower left")
    pp = os.path.join(outdir, f"{prefix}_pr.png")
    plt.savefig(pp, dpi=150, bbox_inches="tight"); plt.close()
    return {"roc_auc": float(auc), "average_precision": float(ap), "roc_path": rp, "pr_path": pp}

def train_and_eval(X_train, y_train, X_test, y_test, outdir, rf_trees=200, n_repeats_pi=3):
    # Logistic Regression
    logit = Pipeline(steps=[
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(max_iter=1500, class_weight="balanced", solver="liblinear", random_state=42))
    ])
    logit.fit(X_train, y_train)
    lp = logit.predict_proba(X_test)[:,1]
    lm = plot_roc_pr(y_test, lp, "logistic", outdir)
    pi_logit = permutation_importance(logit, X_test, y_test, n_repeats=n_repeats_pi, random_state=42, scoring="roc_auc")
    pd.DataFrame({"feature": X_test.columns, "importance_mean": pi_logit.importances_mean, "importance_std": pi_logit.importances_std})\
        .sort_values("importance_mean", ascending=False)\
        .to_csv(os.path.join(outdir, "logistic_permutation_importance.csv"), index=False)

    # Random Forest
    rf = RandomForestClassifier(n_estimators=rf_trees, n_jobs=-1, class_weight="balanced_subsample", random_state=42)
    rf.fit(X_train, y_train)
    rp = rf.predict_proba(X_test)[:,1]
    rm = plot_roc_pr(y_test, rp, "randomforest", outdir)
    pi_rf = permutation_importance(rf, X_test, y_test, n_repeats=n_repeats_pi, random_state=42, scoring="roc_auc")
    pd.DataFrame({"feature": X_test.columns, "importance_mean": pi_rf.importances_mean, "importance_std": pi_rf.importances_std})\
        .sort_values("importance_mean", ascending=False)\
        .to_csv(os.path.join(outdir, "rf_permutation_importance.csv"), index=False)

    return {"logistic": lm, "random_forest": rm}

def main(args):
    os.makedirs(args.outdir, exist_ok=True)
    df = pd.read_csv(args.input)
    assert TARGET in df.columns, f"Target '{TARGET}' not found."
    if ID_COL not in df.columns:
        df[ID_COL] = np.arange(len(df))


    train_df, test_df = stratified_split(df, TARGET, TEST_SIZE, RANDOM_STATE)
    for col in ["MechVent"]:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(0).astype(int)
            test_df[col]  = test_df[col].fillna(0).astype(int)

    LEAKY_COLS = {
        "Survival",
        "Length_of_stay",
        "ICU_length_of_stay",
        "Readmission",
        "hospital_expire_flag",
    }
    def is_leaky(col: str) -> bool:
        low = col.lower()
        bad_tokens = [
            "survival","death","expired","discharge",
            "los","length_of_stay","mortality","expire"
        ]
        return any(tok in low for tok in bad_tokens)

    BASE_EXCLUDE = {TARGET, ID_COL}

    feature_cols = [
        c for c in train_df.columns
        if c not in BASE_EXCLUDE
        and c not in LEAKY_COLS
        and not is_leaky(c)
    ]

    ms = compute_missing_signal(train_df[feature_cols + [TARGET]], feature_cols, TARGET)
    ms.to_csv(os.path.join(args.outdir, "missing_signal_train.csv"), index=False)

    train_df, test_df, high_missing_feats = add_missing_flags(
        train_df, test_df, feature_cols, args.missing_threshold
    )
    train_df, test_df, medians = impute_train_test(train_df, test_df, feature_cols)

    allowed_features = feature_cols + [
        f"{c}_missing" for c in high_missing_feats
        if f"{c}_missing" in train_df.columns
    ]
    train_out = train_df[[ID_COL, TARGET] + allowed_features].copy()
    test_out  = test_df [[ID_COL, TARGET] + allowed_features].copy()

    X_train = train_out.drop(columns=[ID_COL, TARGET])
    X_test  = test_out.drop(columns=[ID_COL, TARGET])
    leaks_in_X = [c for c in X_train.columns if c in LEAKY_COLS or is_leaky(c)]
    if leaks_in_X:
        raise ValueError(f"Leaky columns still present: {leaks_in_X}")

    train_out.to_csv(os.path.join(args.outdir, "train_flags.csv"), index=False)
    test_out.to_csv(os.path.join(args.outdir, "test_flags.csv"), index=False)
    metrics = train_and_eval(
        X_train, train_out[TARGET].values,
        X_test,  test_out[TARGET].values,
        args.outdir, rf_trees=args.rf_trees, n_repeats_pi=args.n_repeats_pi
    )

    summary = {
        "missing_threshold": args.missing_threshold,
        "high_missing_features": high_missing_feats,
        "train_shape": list(train_out.shape),
        "test_shape": list(test_out.shape),
        "class_rate_train": float(train_out[TARGET].mean()),
        "class_rate_test": float(test_out[TARGET].mean()),
        "logistic": metrics["logistic"],
        "random_forest": metrics["random_forest"],
        "outputs": {
            "train_csv": "train_with_flags.csv",
            "test_csv": "test_with_flags.csv",
            "missing_signal": "missing_signal_train.csv",
            "logistic_perm": "logistic_permutation_importance.csv",
            "rf_perm": "rf_permutation_importance.csv",
            "logistic_roc_png": "logistic_roc.png",
            "logistic_pr_png": "logistic_pr.png",
            "rf_roc_png": "randomforest_roc.png",
            "rf_pr_png": "randomforest_pr.png"
        }
    }
    with open(os.path.join(args.outdir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
def export_selected_features(outdir: str,
                             topk_rf: int = 25,
                             topk_logit: int = 25,
                             min_importance: float = 0.0,
                             clinical_keep: List[str] = None):
    """
    基于置换重要性进行特征选择，并导出:
      - final_selected_features.csv   (只有特征名)
      - train_selected.csv / test_selected.csv (ID + TARGET + 选中特征)
    选择规则:
      1) RF & Logistic 各自按 importance_mean 排序取 Top-K
      2) 并集，再过滤 importance_mean >= min_importance（如果设定）
      3) 与 clinical_keep 合并（人工保底）
    """
    if clinical_keep is None:
        clinical_keep = [
            # 建议保底的 ICU 关键特征
            "Age","GCS","SOFA","MAP","HR","RespRate","SaO2","FiO2",
            "BUN","Creatinine","Lactate","WBC","MechVent"
        ]


    rf_path = os.path.join(outdir, "rf_permutation_importance.csv")
    lg_path = os.path.join(outdir, "logistic_permutation_importance.csv")
    rf_imp = pd.read_csv(rf_path) if os.path.exists(rf_path) else pd.DataFrame(columns=["feature","importance_mean"])
    lg_imp = pd.read_csv(lg_path) if os.path.exists(lg_path) else pd.DataFrame(columns=["feature","importance_mean"])


    rf_top = rf_imp.sort_values("importance_mean", ascending=False)
    rf_top = rf_top[rf_top["importance_mean"] >= min_importance].head(topk_rf)["feature"]
    lg_top = lg_imp.sort_values("importance_mean", ascending=False)
    lg_top = lg_top[lg_top["importance_mean"] >= min_importance].head(topk_logit)["feature"]

    selected = set(rf_top).union(set(lg_top)).union(set(clinical_keep))
    train_p = os.path.join(outdir, "train_flags.csv")
    test_p  = os.path.join(outdir, "test_flags.csv")
    train_df = pd.read_csv(train_p)
    test_df  = pd.read_csv(test_p)

    selected_present = [c for c in selected if c in train_df.columns]

    pd.Series(selected_present, name="feature").to_csv(
        os.path.join(outdir, "final_selected_features.csv"),
        index=False, header=False
    )

    keep_cols = [ID_COL, TARGET] + selected_present
    train_df[keep_cols].to_csv(os.path.join(outdir, "train_selected.csv"), index=False)
    test_df[keep_cols].to_csv(os.path.join(outdir, "test_selected.csv"), index=False)
    OUTDIR = r"D:/project/icu_outputs11"   
    export_selected_features(
    outdir=OUTDIR,
    topk_rf=25,
    topk_logit=25,
    min_importance=0.0,  
    )

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=str, default="D:/project/icu_features.csv")
    parser.add_argument("--outdir", type=str, default="D:/project/icu_outputs11")
    parser.add_argument("--missing-threshold", type=float, default=0.50)
    parser.add_argument("--rf-trees", type=int, default=200)
    parser.add_argument("--n-repeats-pi", type=int, default=3)
    args, _ = parser.parse_known_args()
    main(args)